# Inspect Aggregated ERA5-Land SWE (`sd`)

HRU-level inspection of ERA5-Land daily snow depth water equivalent. Companion to the `era5_land_sd` adapter in `aggregate/era5_land.py` (the `sd` variable lives in its own adapter so the runoff side of ERA5-Land — ro/sro/ssro — can keep its monthly cadence and accumulated-flux dispatch separate).

Source (see `catalog/sources.yml` → `era5_land` → `sd`):

- ERA5-Land daily `sd` (snow depth water equivalent), units `m` of water equivalent, `cell_methods: "time: point"` (instantaneous daily snapshot, NOT accumulated like ro/sro/ssro).
- Consolidated to daily under `<datastore>/era5_land/daily/era5_land_daily_<year>.nc` via `.mean()` over the 24 hourly snapshots per day (instantaneous reducer; never the diff-of-accumulations sum used for runoff).
- Aggregated to `<project>/data/aggregated/era5_land_sd/era5_land_sd_<year>_agg.nc`.

Per the repo's transformation policy (`docs/architecture/transformation-pipeline.md`): the aggregator preserves native `m` units and uses `stat_method="mean"` (no per-pixel masking on the ERA5-Land side). The unit chain to PRMS `pkwater_equiv` inches happens **post-aggregation** — this notebook applies `× 1000` (m → mm) then `÷ 25.4` (mm → inches) inline for display, and `targets/swe.py` applies the same chain for the final calibration target.

## Per-target conventions in this notebook

- The aggregated NC carries `sd` in its **native `m`** units (catalog says `cf_units: "m"`). The on-disk values arrive as small floats — typical winter mountain HRUs land between 0.05 m and ~3 m water equivalent.
- This notebook converts to **mm** (`× 1000`) and **inches** (`÷ 25.4`) for sanity-check panels; the underlying values remain in m.
- `sd` is **instantaneous** (`cell_methods: "time: point"`) — a monthly aggregate is the mean over time, not a sum. The instantaneous semantics is read from the catalog at fetch time via `fetch/era5_land.py:_derive_variable_kind()` (refactored to be catalog-driven in PR #146); a future flip of the catalog's `cell_methods` would propagate through the pipeline automatically.
- ERA5-Land's native grid is 0.1° (~11 km). It is the **coarsest** of the four SWE sources — expect smoother spatial maps than SNODAS (1 km) and Daymet (1 km), and missing fine-scale ridge/valley contrast in complex terrain.
- Single-source-in-isolation view here. The cross-source picture is in `inspect_target_swe.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/aggregated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

# The on-disk source_key (synthetic disambiguation per PR #135's SHIM
# convention — the SHIM declares `source_key="era5_land_sd"` so the sd
# aggregated NCs live in their own subdir distinct from the runoff side).
SOURCE_KEY = "era5_land_sd"
# The catalog source key (the canonical name `era5_land`, which holds
# the variable entry for `sd` with cf_units="m").
CATALOG_SOURCE_KEY = "era5_land"
VAR = "sd"
TARGET_YEAR = 2010
TARGET_DAY = "2010-02-15"
TARGET_MONTH = 2  # February
TIME_SERIES_YEARS = range(2005, 2011)
# m of water → mm → inches (PRMS pkwater_equiv).
MM_PER_M = 1000.0
MM_PER_INCH = 25.4

REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Northern Rockies (MT/WY)": (-110.5, 45.5),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated dataset

Opens `<project>/data/aggregated/era5_land_sd/era5_land_sd_<TARGET_YEAR>_agg.nc`. The aggregated NC is per-calendar-year; if the year hasn't been aggregated yet, the cell prints a clear skip and remaining cells fall through gracefully.

Note the **two source keys**: `era5_land_sd` is the on-disk storage key (synthetic), `era5_land` is the catalog source key. Look up units against the catalog key; look up files against the storage key.

In [ ]:
paths = discover_aggregated(project_dir, SOURCE_KEY)
if paths is None:
    print(
        f"SKIP {SOURCE_KEY}: no aggregated NCs at "
        f"{project_dir}/data/aggregated/{SOURCE_KEY}/. Run "
        f"`pixi run nhf-targets agg era5-land-sd --project-dir {project_dir}` first."
    )
    ds = None
else:
    ds = open_year(project_dir, SOURCE_KEY, TARGET_YEAR)
    units = unit_from_catalog(CATALOG_SOURCE_KEY, VAR)
    values_t0 = ds[VAR].isel(time=0).to_pandas()
    print(
        f"ERA5-Land sd: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get(fabric_cfg['id_col'], 'unknown')} | "
        f"NaN HRUs (t=0): {nan_hru_count(values_t0)} | "
        f"catalog units: {units}"
    )
    display(ds)

## Daily snapshot + monthly mean (m, mm, inches)

Side-by-side: the daily snapshot on `TARGET_DAY` and the monthly mean over `TARGET_YEAR-TARGET_MONTH`. Top row keeps native `m`; bottom row converts to inches (PRMS units). Colour scale uses the 95th percentile of the daily snapshot so the deepest-pack HRUs don't flatten everything else.

In [ ]:
if ds is None:
    print("No aggregated ERA5-Land sd; skipping panels.")
else:
    da_day_m = ds[VAR].sel(time=TARGET_DAY, method="nearest")
    monthly_m = (
        ds[VAR]
        .sel(
            time=slice(
                pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1),
                pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1)
                + pd.offsets.MonthEnd(0),
            )
        )
        .mean("time", skipna=True)
        .to_pandas()
    )
    day_m = da_day_m.to_pandas()
    day_in = (day_m * MM_PER_M) / MM_PER_INCH
    monthly_in = (monthly_m * MM_PER_M) / MM_PER_INCH

    vmax_m = float(np.nanpercentile(day_m.dropna(), 95))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), squeeze=False)
    plot_hru_choropleth(
        axes[0, 0], fabric, day_m,
        vmin=0, vmax=vmax_m, cmap="Blues",
        title=f"Daily sd\n{TARGET_DAY}",
        units="m (water equiv)",
    )
    plot_hru_choropleth(
        axes[0, 1], fabric, monthly_m,
        vmin=0, vmax=vmax_m, cmap="Blues",
        title=f"Monthly mean sd\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="m (water equiv)",
    )
    plot_hru_choropleth(
        axes[1, 0], fabric, day_in,
        vmin=0, vmax=(vmax_m * MM_PER_M) / MM_PER_INCH, cmap="Blues",
        title=f"Daily sd (PRMS units)\n{TARGET_DAY}",
        units="inches",
    )
    plot_hru_choropleth(
        axes[1, 1], fabric, monthly_in,
        vmin=0, vmax=(vmax_m * MM_PER_M) / MM_PER_INCH, cmap="Blues",
        title=f"Monthly mean sd (PRMS units)\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="inches",
    )
    fig.suptitle(
        "ERA5-Land sd on HRU fabric — aggregator preserves native m; "
        "PRMS chain (× 1000 → mm, ÷ 25.4 → in) applied for display",
        fontsize=12, y=1.00,
    )
    plt.tight_layout()
    save_figure(fig, "era5_land_sd_native_units_map")
    plt.show()

### Reading the panels

- **Top row (m).** The daily snapshot reveals ERA5-Land's coarse resolution — the western mountain HRUs show a smoother gradient than SNODAS or Daymet at the same date. The monthly mean is broadly the same pattern with mid-month melt/redeposit averaged out.
- **Bottom row (inches).** This is the unit `targets/swe.py` ultimately emits. Same visual pattern as the top row (linear rescale); the inches scale should run roughly **0 → ~40 in** for a typical winter month, vs **0 → ~1.0 m** in the native panel (1 m ≈ 39.4 in).
- **Red flags.** Native values that exceed ~5 m (≈ 200 in) on more than a handful of HRUs are suspicious — ERA5-Land's coarse cells smooth out the deepest peaks SNODAS sees, so a notebook reading deeper than SNODAS is a smoking gun for a missed `× 1000` somewhere upstream. A flat-zero map in February over the Rockies usually means the `instantaneous` reducer was bypassed in consolidation (the accumulated reducer applied to an instantaneous field would silently produce values near 0).

In [ ]:
if ds is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    monthly_finite_m = monthly_m.dropna()
    ax.hist(monthly_finite_m, bins=60, histtype="step", linewidth=2, density=True)
    ax.set_xlabel("sd (m water-eq, monthly mean)")
    ax.set_ylabel("Density")
    ax.set_title(
        f"HRU-level monthly-mean sd distribution — {TARGET_YEAR}-{TARGET_MONTH:02d}"
    )
    save_figure(fig, "era5_land_sd_histogram")
    plt.show()
    print(
        f"monthly mean sd summary (m): "
        f"min={monthly_finite_m.min():.4f}, "
        f"median={monthly_finite_m.median():.4f}, "
        f"mean={monthly_finite_m.mean():.4f}, "
        f"95th={np.nanpercentile(monthly_finite_m, 95):.4f}, "
        f"max={monthly_finite_m.max():.4f}"
    )

### Reading the sd histogram

Expected shape for a winter month is **right-skewed with a heavy zero peak**:

- A tall spike at **0 m** — most HRUs (lower elevation / southern CONUS) carry essentially no snow in any given winter month.
- A long right tail — mountain HRUs reach 0.2–1.5 m water-eq in peak winter; the deepest Sierra and Cascades HRUs can briefly exceed 2 m on short timescales but average lower over a month at ERA5-Land's coarse resolution.
- A near-continuous distribution between, populated by mid-elevation and northern-CONUS HRUs.

Useful red flags:

- **No HRUs above ~0.05 m in February anywhere on the Rockies / Cascades.** Either the consolidated NC missed the deep-pack regions, or the `instantaneous` reducer didn't fire and the accumulated reducer silently zeroed everything out.
- **A spike at a non-zero round value.** Means the `_FillValue` handling failed and an integer fill is being treated as data.
- **A clean Gaussian distribution.** Wrong — sd is bimodal (snowy vs not) at the HRU scale in winter.

In [ ]:
if ds is not None:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    ds_range = open_year_range(project_dir, SOURCE_KEY, TIME_SERIES_YEARS)
    try:
        id_dim = fabric_cfg["id_col"]
        sd_full_m = (
            ds_range[VAR]
            .sel({id_dim: list(rep_hrus.values())})
            .load()
        )
    finally:
        ds_range.close()

    sd_full_in = (sd_full_m * MM_PER_M) / MM_PER_INCH

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        ax.plot(
            sd_full_in.time,
            sd_full_in.sel({id_dim: hru_id}).values,
            linewidth=0.6,
        )
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("sd (inches)")
        ax.grid(alpha=0.3)
    fig.suptitle(
        f"ERA5-Land sd at representative HRUs — {min(TIME_SERIES_YEARS)}–"
        f"{max(TIME_SERIES_YEARS)} (daily, PRMS units)"
    )
    plt.tight_layout()
    save_figure(fig, "era5_land_sd_time_series")
    plt.show()

### Representative-HRU seasonality check

Each panel is a multi-year daily sd time series at one HRU. Expected signatures (in inches, PRMS units):

- **Olympic Peninsula / Northern Rockies (mountain HRUs)** — Strong winter peaks (Dec–Apr), with the Olympics often topping 20–60 in mid-winter and the Northern Rockies typically 10–30 in. Slow spring melt curve through May–Jun, near zero Jul–Oct. **Compared to SNODAS, ERA5-Land's coarse 11 km cells will systematically under-resolve the deepest peaks** — a mountain HRU that touches a single SNODAS deep-snow pixel may register ~80 in there, but only ~30 in in ERA5-Land. This is structural, not a bug.
- **Iowa cropland (Midwest)** — Sharper winter peaks (Jan–Feb) usually under ~5 in, with rapid early-spring melt and zero pack May–Oct.
- **Southern Appalachians (Eastern forest)** — Short snow pulses lasting days to a week, small amplitude (< 2 in typically). The trace should look like a series of spikes rather than a continuous winter pack.

**Red flags.** A flat-zero line on a mountain HRU is a sign of the `instantaneous` reducer being bypassed (try `fetch/era5_land.py:_derive_variable_kind()` — the test added in PR #146 should have caught that). A constant non-zero baseline year-round means a missed time index alignment somewhere upstream.

In [ ]:
if ds is not None:
    n_nan = nan_hru_count(monthly_m)
    print(
        f"ERA5-Land sd {TARGET_YEAR}-{TARGET_MONTH:02d}: "
        f"{n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)"
    )
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    plot_nan_hrus(
        axes[0, 0],
        fabric,
        monthly_m,
        title=(
            f"NaN HRUs (red) — {n_nan} of {len(fabric)} "
            f"(monthly mean, {TARGET_YEAR}-{TARGET_MONTH:02d})"
        ),
    )
    fig.suptitle(
        "Coverage gaps — typically minimal; ERA5-Land covers globe so no CONUS-edge mask",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, "era5_land_sd_coverage")
    plt.show()

### Coverage-gap interpretation

Unlike SNODAS (CONUS-masked) and Margulis WUS-SR (Western US only), **ERA5-Land is global** — the bbox for the fetch step encloses CONUS + contributing watersheds in Canada and Mexico, so HRUs along the CONUS edge get a valid neighbour pixel from the same source. NaN coverage at HRU level should therefore be **negligible**: anything more than a stray edge HRU is a fetch/consolidation regression.

Two failure modes to watch for:

- **A consistent NaN band along one edge.** Suggests the consolidation pipeline silently truncated the bbox (e.g. a CDS area downgrade to the requested core CONUS instead of the buffered CONUS+).
- **Random scattered NaN HRUs.** Suggests dask chunking + a missing time-axis alignment between hourly chunks and the daily resample — extremely unusual but worth flagging if it appears.

**Downstream handling.** Like all four SWE sources, ERA5-Land enters the target via NaN-aware min/max in `targets/swe.py`. An HRU that's NaN here can still get a finite bound from Daymet / SNODAS / Margulis if any of those carry data there.

In [ ]:
if ds is not None:
    # Magnitude sanity check: HRU-area-weighted monthly mean (m) over the
    # fabric. Published CONUS-mean winter peak ERA5-Land sd lands around
    # 0.02–0.04 m water equivalent (≈ 20–40 mm); this is in the same
    # ballpark as SNODAS, but ERA5-Land tends to underestimate the deepest
    # mountain HRUs because of resolution.
    # Memory: feedback_validate_magnitudes — |% diff| > 30% from a published
    # mean is a smoking gun for a missed conversion factor.
    agg_mean_m = area_weighted_mean(monthly_m, fabric)
    agg_mean_in = (agg_mean_m * MM_PER_M) / MM_PER_INCH if agg_mean_m == agg_mean_m else float("nan")
    print(
        f"ERA5-Land sd {TARGET_YEAR}-{TARGET_MONTH:02d} HRU-area-weighted mean: "
        f"{agg_mean_m:.5f} m  ({agg_mean_m * MM_PER_M:.2f} mm; {agg_mean_in:.3f} inches)"
    )
    print(
        "Expected order of magnitude for CONUS winter peak: tens of mm "
        "(comparable to SNODAS); a result an order of magnitude off (e.g. "
        "single mm or single m) is a smoking gun for the m↔mm conversion."
    )

## Why this is a single-source diagnostic, not a cross-source comparison

ERA5-Land `sd` is one of four sources for the SWE calibration target. This notebook checks that the ERA5-Land-side aggregation is sane on its own; the cross-source view (with Daymet, SNODAS, and Margulis WUS-SR alongside) lives in `inspect_target_swe.ipynb`.

- **No quality gate at aggregate time.** ERA5-Land emits clean floats with no per-pixel mask. The aggregator uses `stat_method="mean"` with no `pre_aggregate_hook`.
- **Units pass through.** The aggregated NC preserves native `m`. The `× 1000` (m → mm) and `÷ 25.4` (mm → in) rescales are the notebook's / target's job; CLAUDE.md's transformation policy reserves linear unit conversions for `targets/`.
- **Time is point, not sum.** sd is `cell_methods: "time: point"`. A monthly aggregate is the mean over daily instantaneous values, not a sum of accumulations — exactly like SNODAS. The instantaneous-vs-accumulated dispatch in `fetch/era5_land.py` is keyed on `cell_methods` read from the catalog (PR #146), so a future flip would propagate automatically.
- **Coarse resolution caveat.** ERA5-Land's 0.1° (~11 km) cells are 10× larger than SNODAS/Daymet (~1 km), so the deepest-pack HRUs systematically read lower here than in SNODAS. This is a real feature of the source — keep it in the SWE target as the lower bound on the deep-pack HRUs.

In [ ]:
if ds is not None:
    ds.close()